In [ ]:
import math

SKILL_ALIASES = {
    'python': 'python', 'pyhton': 'python',
    'java': 'java',
    'javascript': 'javascript', 'javascrpit': 'javascript', 'js': 'javascript',
    'typescript': 'typescript', 'typescrpit': 'typescript',
    'c++': 'cpp', 'cpp': 'cpp', 'r': 'r', 'kotlin': 'kotlin',
    'machinelearning': 'machine_learning', 'machine learning': 'machine_learning',
    'ml': 'machine_learning', 'sklearn': 'machine_learning',
    'deeplearning': 'deep_learning', 'deep learning': 'deep_learning', 'deep-learning': 'deep_learning',
    'tensorflow': 'tensorflow', 'pytorch': 'pytorch', 'keras': 'keras',
    'nlp': 'nlp', 'bert': 'bert', 'xgboost': 'xgboost',
    'feature engineering': 'feature_engineering',
    'statistics': 'statistics', 'stats': 'statistics',
    'regression': 'regression', 'clustering': 'clustering',
    'data-viz': 'data_visualization', 'data visualization': 'data_visualization',
    'matplotlib': 'data_visualization', 'tableau': 'data_visualization',
    'power-bi': 'data_visualization', 'powerbi': 'data_visualization',
    'pandas': 'pandas', 'numpy': 'numpy',
    'react': 'react', 'reacts': 'react', 'reactjs': 'react',
    'vue': 'vue', 'vue.js': 'vue',
    'redux': 'redux', 'tailwind': 'tailwind',
    'html/css': 'html_css', 'html': 'html_css', 'css': 'html_css',
    'jest': 'jest', 'graphql': 'graphql',
    'node.js': 'nodejs', 'nodejs': 'nodejs',
    'flask': 'flask',
    'spring boot': 'spring_boot', 'springboot': 'spring_boot',
    'rest api': 'rest_api', 'rest': 'rest_api',
    'microservices': 'microservices',
    'sql': 'sql', 'mysql': 'mysql',
    'postgresql': 'postgresql', 'postgres': 'postgresql',
    'mongodb': 'mongodb', 'redis': 'redis',
    'docker': 'docker', 'kubernetes': 'kubernetes', 'kubernates': 'kubernetes',
    'ci/cd': 'ci_cd', 'aws': 'aws',
    'android': 'android', 'firebase': 'firebase',
    'algorithms': 'algorithms', 'algoritms': 'algorithms',
    'data structure': 'data_structures', 'data structures': 'data_structures',
    'competitive programming': 'competitive_programming',
    'ui/ux': 'ui_ux', 'figma': 'figma'
}

RAW_RESUMES = {
    'Arjun Sharma':    'Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning',
    'Priya Nair':      'JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS',
    'Rahul Gupta':     'Java, Spring Boot, MySql, Microservices, Docker, kubernates',
    'Sneha Patel':     'Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib',
    'Vikram Singh':    'C++, Algoritms, Data Structure, competitive programming, python',
    'Ananya Krishnan': 'javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD',
    'Karan Mehta':     'Python, Sklearn, XGboost, feature engineering, SQL, tableau',
    'Deepika Rao':     'Java, Android, Kotlin, Firebase, REST, UI/UX, figma',
    'Aditya Kumar':    'Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest',
    'Meera Iyer':      'python, R, statistics, ML, regression, clustering, Power-BI'
}

JD_SKILLS = {
    'JD1': ['python','machine_learning','deep_learning','tensorflow','pytorch','sql','data_visualization','nlp','bert','feature_engineering','statistics'],
    'JD2': ['java','spring_boot','mysql','postgresql','microservices','docker','kubernetes','rest_api','ci_cd','redis'],
    'JD3': ['javascript','react','vue','typescript','rest_api','html_css','nodejs','graphql','redux','jest','aws']
}

JD_LABELS = {
    'JD1': 'JD-1 — Kakao (ML Engineer)',
    'JD2': 'JD-2 — Naver (Backend Engineer)',
    'JD3': 'JD-3 — Line (Frontend Engineer)'
}

def normalize(raw):
    tokens = [t.strip().lower() for t in raw.split(',')]
    canonical = []
    for token in tokens:
        matched = False
        for key in sorted(SKILL_ALIASES.keys(), key=len, reverse=True):
            if ' ' in key and token == key:
                canonical.append(SKILL_ALIASES[key])
                matched = True
                break
        if not matched and token in SKILL_ALIASES:
            canonical.append(SKILL_ALIASES[token])
    seen = set()
    result = []
    for s in canonical:
        if s not in seen:
            seen.add(s)
            result.append(s)
    return result

resumes = {name: normalize(raw) for name, raw in RAW_RESUMES.items()}
vocab = sorted(set(s for skills in resumes.values() for s in skills))
df = {skill: sum(1 for skills in resumes.values() if skill in skills) for skill in vocab}

def tfidf_vector(skills):
    N = len(skills)
    return [((1/N) * math.log(10/df[s])) if s in skills else 0.0 for s in vocab]

def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(x*x for x in b))
    return dot/(na*nb) if na and nb else 0.0

tfidf = {name: tfidf_vector(skills) for name, skills in resumes.items()}
jd_vecs = {jd: [1 if s in skills else 0 for s in vocab] for jd, skills in JD_SKILLS.items()}

for jd_id, jd_vec in jd_vecs.items():
    scores = sorted([(name, cosine(vec, jd_vec)) for name, vec in tfidf.items()], key=lambda x: (-x[1], x[0]))
    print(JD_LABELS[jd_id])
    print(', '.join(f'{name}({score:.2f})' for name, score in scores[:3]))
    print()